# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [1]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [2]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from lib.vector_db import VectorStoreManager, CorpusLoaderService
from lib.rag import RAG
from lib.llm import LLM

In [3]:
import os
from typing import List
from dotenv import load_dotenv

from lib.agents import Agent
from lib.llm import LLM
from lib.state_machine import Run
from lib.messages import BaseMessage
from lib.tooling import tool
from lib.vector_db import VectorStoreManager, CorpusLoaderService
from lib.rag import RAG

from lib.messages import BaseMessage
from lib.tooling import tool
from typing import List


In [4]:
# TODO: Create a .env file with the following variables
# OPENAI_API_KEY="YOUR_KEY"
# CHROMA_OPENAI_API_KEY="YOUR_KEY"
# TAVILY_API_KEY="YOUR_KEY"

In [5]:
# TODO: Load environment variables
import os
load_dotenv(dotenv_path="config.env")
print(os.getenv("OPENAI_API_KEY"))
print(os.getenv("OPENAI_BASE_URL"))
open_api_key = os.getenv("OPENAI_API_KEY")

voc-71328759916886552416536a5a3865718728.78066438
https://openai.vocareum.com/v1


### VectorDB Instance

In [6]:
# TODO: Instantiate your ChromaDB Client
# Choose any path you want
#chroma_client = chromadb.PersistentClient(path="chromadb")
#Reused the VectorStoreManager class with persist db

### Collection

In [7]:
# TODO: Pick one embedding function
# If picking something different than openai, 
# make sure you use the same when loading it
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(api_key=open_api_key)


In [8]:
# TODO: Create a collection
# Choose any name you want
vector_store_manager = VectorStoreManager(openai_api_key=open_api_key)
vector_store = vector_store_manager.get_or_create_store("udaplay")

### Add documents

In [9]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    content = f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - {game['Description']}"
    doc_id = os.path.splitext(file_name)[0]

    vector_store._collection.add(
        ids=[doc_id],
        documents=[content],
        metadatas=[game]
    )

In [10]:
print(vector_store._collection.count())

15


In [11]:
query = "Who developed Gran Turismo?"
results = vector_store._collection.query(
    query_texts=[query],
    n_results=3,
    include=["documents", "metadatas", "distances"],
)

print("Query:", query)
for doc, meta, dist in zip(
    results["documents"][0],
    results["metadatas"][0],
    results["distances"][0],
):
    print(f"- {meta.get('Name')} | {meta.get('Platform')} | {meta.get('YearOfRelease')} | distance={dist}")
    print(f"  {doc}")

Query: Who developed Gran Turismo?
- Gran Turismo | PlayStation 1 | 1997 | distance=0.26645877957344055
  [PlayStation 1] Gran Turismo (1997) - A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.
- Gran Turismo 5 | PlayStation 3 | 2010 | distance=0.27101248502731323
  [PlayStation 3] Gran Turismo 5 (2010) - A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.
- Grand Theft Auto: San Andreas | PlayStation 2 | 2004 | distance=0.44407397508621216
  [PlayStation 2] Grand Theft Auto: San Andreas (2004) - An expansive open-world game set in the fictional state of San Andreas, following the story of Carl 'CJ' Johnson.


### Create RAG pipeline

In [13]:
rag_llm = LLM(
    model="gpt-4o-mini",
    temperature=0.3,
)

In [14]:
games_market_rag = RAG(
    llm=rag_llm,
    vector_store = vector_store
)

### Results

In [15]:
result:Run = games_market_rag.invoke("What are the games published by Sony Computers")
print(result.get_final_state()["answer"])

[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
The games published by Sony Computer Entertainment mentioned in the context are:

1. Gran Turismo 5 (2010) - PlayStation 3
2. Gran Turismo (1997) - PlayStation 1
3. Marvel's Spider-Man (2018) - PlayStation 4


In [16]:
result:Run = games_market_rag.invoke("What are the games which involves saving a queen or kingdom?")
print(result.get_final_state()["answer"])

[StateMachine] Starting: __entry__
[StateMachine] Executing step: retrieve
[StateMachine] Executing step: augment
[StateMachine] Executing step: generate
[StateMachine] Terminating: __termination__
The games that involve saving a queen or kingdom from the provided context are:

1. **Super Mario World (1990)** - Mario embarks on a quest to save Princess Toadstool and Dinosaur Land.
2. **Super Mario 64 (1996)** - Mario's quest is to rescue Princess Peach.
